In [9]:
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- 1. Data Loading and Initial Cleaning ---

print("--- 1. Data Loading and Cleaning Start ---")

# Define the folder path prefix to locate the CSV files
FOLDER_PATH = 'datasets/' 

# List of files to load
file_names = {
    'field_crop': FOLDER_PATH + 'Field_Crop_UttarPradesh.csv',
    'commercial_trees': FOLDER_PATH + 'Commercial_trees_UttarPradesh.csv',
    'flori': FOLDER_PATH + 'flori.csv',
    'crp_aa': FOLDER_PATH + 'Crp_aa_dataset.csv'
}

# Load the datasets with error handling
try:
    # Commercial_trees.csv is messy, header=1 skips the unwanted first row.
    df_commercial_trees = pd.read_csv(file_names['commercial_trees'], header=1) 
    
    # Load other datasets
    df_field_crop = pd.read_csv(file_names['field_crop'])
    df_flori = pd.read_csv(file_names['flori'])
    df_crp_aa = pd.read_csv(file_names['crp_aa'])

except FileNotFoundError as e:
    print(f"\nFATAL ERROR: {e}")
    print("Please ensure the folder structure is correct: CSV files must be inside a folder named 'datasets' in the same directory as this notebook.")
    # Stop execution if files are missing
    raise

# 1.1 Field Crop (df_field_crop) Cleaning
# Filter for 'Soil, Water & pH' category
df_field_crop = df_field_crop[df_field_crop['Category'] == 'Soil, Water & pH'].copy()
df_field_crop = df_field_crop[['District', 'Crop', 'Requirements']].rename(
    columns={'Requirements': 'Soil_Water_pH'}
)
df_field_crop.dropna(subset=['District', 'Crop'], inplace=True)
print(f"Field Crop Cleaned: {len(df_field_crop)} rows.")

# 1.2 Commercial Trees (df_commercial_trees) Cleaning
# Manually rename the messy header columns (assuming the structure after header=1)
df_commercial_trees.columns = ['District', 'Tree', 'Soil_Water_pH', 'Return_Period_Soil']

# Clean up rows that are remnants of the header structure
df_commercial_trees = df_commercial_trees[~df_commercial_trees['District'].astype(str).str.contains(':', na=False)]

# Focus on required columns
df_commercial_trees = df_commercial_trees[['District', 'Tree', 'Soil_Water_pH']].copy()
df_commercial_trees.dropna(subset=['District', 'Tree'], inplace=True)
print(f"Commercial Trees Cleaned: {len(df_commercial_trees)} rows.")

# 1.3 Floriculture (df_flori) Cleaning
df_flori.rename(columns={
    'Major Commercial Flowers': 'Flower', 
    'Soil and Water Requirement': 'Soil_Water_pH'
}, inplace=True)
df_flori = df_flori[['District', 'Flower', 'Soil_Water_pH']].copy()
df_flori.dropna(subset=['District', 'Flower'], inplace=True)
print(f"Floriculture Cleaned: {len(df_flori)} rows.")

# 1.4 Crop Climate Data (df_crp_aa) Cleaning for ML
df_crp_aa.rename(columns={'label': 'Crop'}, inplace=True)
df_crp_aa.dropna(inplace=True)
print(f"ML Dataset Cleaned: {len(df_crp_aa)} rows.")

print("--- Data Cleaning Complete ---")

# --- 2. Machine Learning Model Training (Field Crop) ---

print("\n--- 2. Training Random Forest Model Start ---")

# Features for the ML model (based on Crp_aa_dataset)
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df_crp_aa[features]
y = df_crp_aa['Crop']

# Encode the target variable 'Crop'
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Classifier Accuracy: {accuracy:.4f} (Used for primary crop ranking)")

# --- 3. Rule-Based Recommendation Data Preparation ---

# 3.1 Define complementary interactions
complementary_interactions = {
    'Neem (Azadirachta indica)': ['Marigold', 'Paddy / Rice'], 
    'Marigold': ['Potato', 'Tomato'], 
    'Eucalyptus (Eucalyptus spp.)': ['Mentha'], 
    'Teak / Sagaun (Tectona grandis)': ['Turmeric'], 
    'Gram / Chickpea': ['Mustard / Rapeseed'], 
    'Paddy / Rice': ['Fish/Prawn (Pisciculture)'], 
}

# 3.2 Define ROI/Return Period Mapping for Trees
roi_map = {
    'Sheesham (Dalbergia sissoo)': '12–20 years (poles earlier at 8–12). High-value hardwood.',
    'Teak / Sagaun (Tectona grandis)': '20–25 years (commercial grades). Premium timber, strong demand.',
    'Eucalyptus (Eucalyptus spp.)': '8–12 years (fast-growing pulp/pole wood). Fast returns, lower per-unit price.',
    'Babool / Babul (Acacia nilotica)': '10–15 years (poles, fuelwood). Good for poles, fuelwood, and soil health.',
    'Neem (Azadirachta indica)': '10–15 years (benefits from medicinal uses and seed oil). Valuable by-products.',
    'Mango (Mangifera indica) — boundary/fruit option': '4–8 years (fruiting varieties). Fruit income much earlier than timber.'
}

# --- 4. Saving Artifacts ---

print("\n--- 4. Saving Model and Data Files ---")

# Prepare the data dictionary to be saved
recommendation_data = {
    'field_crop': df_field_crop,
    'commercial_trees': df_commercial_trees,
    'flori': df_flori,
    'interactions': complementary_interactions,
    'roi_map': roi_map
}

# Save the trained model and LabelEncoder
# These files will be saved in the root project folder, not 'datasets'
with open('rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("SUCCESS: Saved 'rf_model.pkl'.")

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("SUCCESS: Saved 'label_encoder.pkl'.")

# Save the preprocessed dataframes and rules
with open('recommendation_data.pkl', 'wb') as f:
    pickle.dump(recommendation_data, f)
print("SUCCESS: Saved 'recommendation_data.pkl'.")

print("\nAll necessary files generated successfully! Proceed to update and run 'app.py'.")

--- 1. Data Loading and Cleaning Start ---
Field Crop Cleaned: 900 rows.
Commercial Trees Cleaned: 1349 rows.
Floriculture Cleaned: 75 rows.
ML Dataset Cleaned: 2200 rows.
--- Data Cleaning Complete ---

--- 2. Training Random Forest Model Start ---
Random Forest Classifier Accuracy: 0.9932 (Used for primary crop ranking)

--- 4. Saving Model and Data Files ---
SUCCESS: Saved 'rf_model.pkl'.
SUCCESS: Saved 'label_encoder.pkl'.
SUCCESS: Saved 'recommendation_data.pkl'.

All necessary files generated successfully! Proceed to update and run 'app.py'.
